#### The CelesTrack Dataset is very clean already, but mistaked can be happened by the dataset operators
Such as : 
- Missing values
- Duplication in the satellites
- value inconsistencies, etc

## Performing preprocssing on the satellite data
- Handle missing values
- Remove duplicates
- Structure data
- Build preprocessing pipeline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Important thorughout this project
%matplotlib inline


In [ ]:
df = pd.read_csv('../data/raw/gp.csv')

In [ ]:
df.head()

In [ ]:
df.info()

----
### Check for duplicate satellite records

In [ ]:
df[df.duplicated()]

- This dataset is very effective, it does not have any duplicate records.
- But, when building the final pipeline, it is still necessary to handle possible duplicates
- Here is a simple method to reliablely remove duplicates

In [ ]:
df = df.drop_duplicates()

-------
### Handling missing values

In [ ]:
df[df.isnull()]

- By looking at the result of df.info(), there are no missing values acorss the entire dataset.
- But, df[df.isnull()] returning wierd result

  
- original dataset was "full" (had no null values),  df.isnull() mask was all False. 
- When you use this all-False mask for indexing, pandas replaces every single value with NaN, giving you a DataFrame full of NaNs.

In [ ]:
# A better reliable way to check if atleast one row has misisng values or not

rows_with_null = df.isnull().any(axis=1)

In [ ]:
rows_with_null

In [ ]:
df[rows_with_null]

#### As the dataset contains no missing values, still there is a need for handling them

In [ ]:
df.head()

#### We will select those features having 'logical' importance to be completlely NaNs free
Here are the following features and their methods to handle them

1. Object Name -> Fill missing names with 'Unknown'
2. Object ID -> Fill missing IDs with 'Unknown'
3. EPOCH ->  If Feature Deriving a new Feature like Days since launch -> Fill accorindgly, else drop the row
4. MEAN_MOTION,	ECCENTRICITY,	INCLINATION, RA_OF_ASC_NODE,	ARG_OF_PERICENTER,	MEAN_ANOMALY
-> mostly dropping the rows but can be derived from other features
5. NORAD_CAT_ID -> Fill with 'Unknown'
6. ELEMENT_SET_NO -> Fill with Mode
7. REV_AT_EPOCH,	BSTAR
-> Mostly imputation (mean / median)
8. MEAN_MOTION_DOT	-> Fill with (mean / median) or any other method (check carefully as it is imp feature)
9. MEAN_MOTION_DDOT -> Fill with mode (as most of them are 0)
10. BSTAR -> Handle carefully or just drop the rows

- Some other features that may not be used at all
1. EPHEMERIS_TYPE,	CLASSIFICATION_TYPE -> We will still use them by filling by 'mode' for missing values

-----

In [ ]:
# Fill Missing object name
df['OBJECT_NAME'] = df['OBJECT_NAME'].fillna('Unknown')

In [ ]:
# Fill Missing object ID 
df['OBJECT_ID']= df['OBJECT_ID'].fillna('Unknown')

In [ ]:
# handle missing epoch -> For now, we will drop the missing rows, later perform feature engineering to fill it.
df = df.dropna(subset = ['EPOCH'])

In [ ]:
# Fill Missing MEAN_MOTION, ECCENTRICITY, INCLINATION, RA_OF_ASC_NODE, ARG_OF_PERICENTER, MEAN_ANOMALY -> Drop the Nulls for now
# These are very important TLE paramters, filling them with an easay method is not recommended, as wrong value can lead to anomolous results
df = df.dropna(subset = ['MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY'])

In [ ]:
# Fill missing NORAD_CAT_ID -> Fill with 'Unknown'
df['NORAD_CAT_ID']= df['NORAD_CAT_ID'].fillna('Unknown')

In [ ]:
# Fill missing ELEMENT_SET_NO -> Fill with imputation : Mode
df['ELEMENT_SET_NO']= df['ELEMENT_SET_NO'].fillna(df['ELEMENT_SET_NO'].mode())

In [ ]:
# Fill missing REV_AT_EPOCH	 -> This is also a very important feature, missing with mean or median probably is not recommended
# So, we will drop the rows right now, later we will see more secured method

df = df.dropna(subset = ['REV_AT_EPOCH'])

In [ ]:
# Fill missing BSTAR -> Very important feature, so we will drop the rows to avoid anomolous filling

df = df.dropna(subset = ['BSTAR'])

In [ ]:
# Fill missing MEAN_MOTION_DOT, MEAN_MOTION_DDOT -> Same like Rev at epoch and Bstar, drop the rows
df = df.dropna(subset = ['MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT'])

In [ ]:
# Fill missing EPHEMERIS_TYPE, CLASSIFICATION_TYPE -> Fill each one by imputation  : Mode
df['EPHEMERIS_TYPE'] = df['EPHEMERIS_TYPE'].fillna(df['EPHEMERIS_TYPE'].mode())

df['CLASSIFICATION_TYPE'] = df['CLASSIFICATION_TYPE'].fillna(df['CLASSIFICATION_TYPE'].mode())